# Distil OpenBabel bond perception into a small AdjMatSeer — data generation

README benchmark: **48% valid (ML bond prediction) vs 93% valid (OpenBabel)**. So obabel is the
*stronger* labeller here, and obabel cannot run in ONNX / the browser. Imitating it in a small
net is both a quality upgrade and the only way to ship obabel-grade bonds to the JS runtime.

## Geometry source (no FM)

Default `SOURCE = "pairs"`: decode the stored `x1` latents from
`./teacher_pairs/teacher_pairs` (already produced by the **420 EDM at 100 steps**).
That skips another 100-NFE pass. Set `SOURCE = "edm"` only if you want a fresh ancestral
draw from `edm_moi_chembl_15_39.pt` (hidden=420, `DIFFUSION_STEPS=100`) using the same
pair contexts (`n_atoms`, `context`).

## The one detail that decides whether this works

At inference the seer is fed `(elements, dist_mat, adj_mat)` where `adj_mat` is **RDKit's**
`rdDetermineBonds.DetermineConnectivity` guess (inside `canonicalise`), not obabel's. So:

- **input** connectivity = RDKit `DetermineConnectivity` (stored as `conn`)
- **target** bond orders = OpenBabel `PerceiveBondOrders` (stored as `target`)

Both are stored separately. Deriving `conn` from `target` would leak the answer and break
train/test match — at inference only the RDKit guess exists. The net's job is bond *orders*
(plus fixing connectivity mistakes), which is exactly where the current seer is weak.

Atom order: `canonicalise` renumbers atoms, so obabel is run on the **canonicalised** xyz.
OpenBabel preserves xyz atom order, so input and target share one index space.

## Aromatic convention

`AROMATIC_MODE="kekule"` (default) labels ring bonds with their Kekulé order (1/2) and lets
RDKit perceive aromaticity itself — far more robust than emitting bare `BondType.AROMATIC`,
which frequently fails kekulization on generated geometries. Uses only classes 1–3, a subset of
the existing 5-class head, so `redefine_bonds` and the JS decoder keep working unchanged.
Set `"class4"` to reproduce the original convention.

## Output

`bond_pairs/shardXXXX.pt` with `elements (int8)`, `coords (f16)`, `conn (int8)`,
`target (int8)`, `n_atoms (int16)`. ~3.6 kB/molecule, so 400k ≈ 1.4 GB. Feed to
`train_small_seer.ipynb`.

Requires `pip install openbabel-wheel` (or a conda `openbabel`).

In [ ]:
# Colab / fresh box only — skip if the env is already set up.
# !pip install openbabel-wheel rdkit tqdm

In [ ]:
"""Decode teacher-pair x1 (or sample EDM 420 / 100 steps), label bonds with OpenBabel."""
import os
from concurrent.futures import ProcessPoolExecutor
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
from rdkit import Chem, RDLogger
from tqdm import tqdm

RDLogger.DisableLog("rdApp.*")

try:
    from src.mlconfgen.utils import ATOM_DECODER, CONTEXT_NORMS, DIMENSION, MAX_N_NODES
    from src.mlconfgen.utils.mol_utils import prepare_masks, samples_to_rdkit_mol
except ImportError:
    from mlconfgen.utils import ATOM_DECODER, CONTEXT_NORMS, DIMENSION, MAX_N_NODES
    from mlconfgen.utils.mol_utils import prepare_masks, samples_to_rdkit_mol

# --------------------- config ---------------------
SOURCE = "pairs"            # "pairs" = decode stored x1; "edm" = sample 420 EDM @ 100 steps
device = "cuda"             # used only when SOURCE == "edm"
EDM_WEIGHTS = Path("./edm_moi_chembl_15_39.pt")
PAIR_DIR = Path("./teacher_pairs/teacher_pairs")
DIFFUSION_STEPS = 100
HIDDEN_NF = 420
TARGET = 400_000
BATCH_PAIRS = 1024          # xyz decode + CPU label
BATCH_EDM = 32              # 100-step 420 ancestral; raise if VRAM allows
SHARD_SIZE = 10_000
N_WORKERS = max(1, (os.cpu_count() or 4) - 1)
AROMATIC_MODE = "kekule"    # "kekule" | "class4"
DROP_UNSANITIZABLE = True   # keep only labels that give a sanitizable molecule
MIN_ATOMS = 15
NOISE_PRECISION = 1e-5
PAD_TO = MAX_N_NODES
SEED = 7

OUT_DIR = Path("./bond_pairs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = OUT_DIR / "generate.log"

torch.manual_seed(SEED)
assert SOURCE in ("pairs", "edm"), SOURCE


def log(msg):
    line = f"{datetime.now().isoformat(timespec='seconds')}  {msg}"
    print(line)
    with open(LOG_PATH, "a") as f:
        f.write(line + "\n")


# --------------------- worker: canonicalise + obabel label ---------------------
# Self-contained so it does not rely on inherited notebook globals (fork start method).
def _init_worker(aromatic_mode, drop_unsan, min_atoms):
    global _AROM, _DROP, _MIN_AT, _DIM, _np, _Chem, _bond_type_dict, _canonicalise, _ob
    import numpy as _numpy
    from rdkit import Chem as _C
    from rdkit import RDLogger as _L
    _L.DisableLog("rdApp.*")
    try:
        from src.mlconfgen.utils.common import bond_type_dict as _btd
        from src.mlconfgen.utils.common import canonicalise as _can
        from src.mlconfgen.utils.config import DIMENSION as _D
    except ImportError:
        from mlconfgen.utils.common import bond_type_dict as _btd
        from mlconfgen.utils.common import canonicalise as _can
        from mlconfgen.utils.config import DIMENSION as _D
    from openbabel import openbabel as _obab
    _obab.obErrorLog.SetOutputLevel(0)
    _AROM, _DROP, _MIN_AT, _DIM = aromatic_mode, drop_unsan, min_atoms, _D
    _np, _Chem, _bond_type_dict, _canonicalise, _ob = _numpy, _C, _btd, _can, _obab


def _obabel_orders(xyz_block, n):
    """OpenBabel bond perception on an xyz block -> list of (i, j, class 1-4)."""
    conv = _ob.OBConversion()
    conv.SetInFormat("xyz")
    obmol = _ob.OBMol()
    if not conv.ReadString(obmol, xyz_block):
        return None
    if obmol.NumAtoms() != n:
        return None
    out = []
    for b in _ob.OBMolBondIter(obmol):
        i, j = b.GetBeginAtomIdx() - 1, b.GetEndAtomIdx() - 1
        if _AROM == "class4" and b.IsAromatic():
            cls = 4
        else:
            cls = min(max(int(b.GetBondOrder()), 1), 3)
        out.append((i, j, cls))
    return out


def label_xyz(xyz_block):
    """xyz (no bonds) -> (elements, coords, conn, target, n) in canonical atom order."""
    Chem, np = _Chem, _np
    mol = Chem.MolFromXYZBlock(xyz_block)
    if mol is None:
        return None
    try:
        mol = _canonicalise(mol)   # DetermineConnectivity + canonical renumbering
    except Exception:
        return None

    n = mol.GetNumAtoms()
    if n < _MIN_AT or n > _DIM:
        return None

    conn = np.zeros((_DIM, _DIM), dtype=np.int8)
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        conn[i, j] = conn[j, i] = 1

    orders = _obabel_orders(Chem.MolToXYZBlock(mol), n)
    if not orders:
        return None
    target = np.zeros((_DIM, _DIM), dtype=np.int8)
    for i, j, cls in orders:
        target[i, j] = target[j, i] = cls

    if _DROP:
        probe = Chem.RWMol()
        for a in mol.GetAtoms():
            probe.AddAtom(Chem.Atom(a.GetAtomicNum()))
        for i, j, cls in orders:
            probe.AddBond(i, j, _bond_type_dict[cls])
        try:
            Chem.SanitizeMol(probe.GetMol())
        except Exception:
            return None

    elements = np.zeros(_DIM, dtype=np.int8)
    elements[:n] = [a.GetAtomicNum() for a in mol.GetAtoms()]
    coords = np.zeros((_DIM, 3), dtype=np.float16)
    coords[:n] = mol.GetConformer().GetPositions()
    return elements, coords, conn, target, n


# --------------------- decode stored x1 ---------------------
def x1_to_xyz_blocks(x1, n_atoms):
    """Teacher-pair x1 is (B, pad, 11) = coords + type logits, already COM-projected."""
    blocks = []
    x1 = x1.float()
    for i in range(x1.shape[0]):
        n = int(n_atoms[i])
        c = x1[i, :n, :3]
        if n < MIN_ATOMS or not torch.isfinite(c).all():
            continue
        types = x1[i, :n, 3:].argmax(-1)
        xyz = f"{n}\n\n"
        for j in range(n):
            el = ATOM_DECODER[int(types[j])]
            xyz += f"{el} {float(c[j, 0]):.9f} {float(c[j, 1]):.9f} {float(c[j, 2]):.9f}\n"
        blocks.append(xyz)
    return blocks


def load_pair_shards():
    paths = sorted(PAIR_DIR.glob("*shard*.pt"))
    if not paths:
        raise FileNotFoundError(f"no *shard*.pt in {PAIR_DIR}")
    packs = [torch.load(p, map_location="cpu", weights_only=False) for p in paths]
    n_atoms = torch.cat([p["n_atoms"].to(torch.int16) for p in packs])
    teacher_steps = int(packs[0].get("teacher_steps", DIFFUSION_STEPS))
    if SOURCE == "pairs":
        x1 = torch.cat([p["x1"] for p in packs])
        ctx = None
    else:
        x1 = None
        ctx = torch.cat([p["context"].float() for p in packs])
    log(f"loaded {len(paths)} pair shards  n={n_atoms.shape[0]}  teacher_steps={teacher_steps}")
    return x1, n_atoms, ctx, teacher_steps


def flush_shard(buf, shard_i, source_name, nfe):
    path = OUT_DIR / f"shard{shard_i:04d}.pt"
    torch.save(
        {
            "elements": torch.from_numpy(np.stack(buf["elements"])),
            "coords": torch.from_numpy(np.stack(buf["coords"])),
            "conn": torch.from_numpy(np.stack(buf["conn"])),
            "target": torch.from_numpy(np.stack(buf["target"])),
            "n_atoms": torch.tensor(buf["n_atoms"], dtype=torch.int16),
            "aromatic_mode": AROMATIC_MODE,
            "source": source_name,
            "nfe": nfe,
        },
        path,
    )
    log(f"wrote {path.name}  n={len(buf['n_atoms'])}")


# --------------------- optional EDM sampler ---------------------
edm = norms = None
if SOURCE == "edm":
    try:
        from src.mlconfgen.egnn import EGNNDynamics
        from src.mlconfgen.equivariant_diffusion import EquivariantDiffusion, PredefinedNoiseSchedule
    except ImportError:
        from mlconfgen.egnn import EGNNDynamics
        from mlconfgen.equivariant_diffusion import EquivariantDiffusion, PredefinedNoiseSchedule

    edm_ckpt = torch.load(EDM_WEIGHTS, map_location="cpu", weights_only=False)
    norms = {
        k: torch.tensor(v, device=device, dtype=torch.float32)
        for k, v in edm_ckpt.get("context_norms", CONTEXT_NORMS).items()
    }
    dyn = EGNNDynamics(in_node_nf=9, context_node_nf=3, hidden_nf=HIDDEN_NF, device=device)
    edm = EquivariantDiffusion(dyn, in_node_nf=8, timesteps=1000, noise_precision=NOISE_PRECISION)
    edm.load_state_dict(edm_ckpt["state_dict"])
    edm.gamma = PredefinedNoiseSchedule(timesteps=DIFFUSION_STEPS, precision=NOISE_PRECISION)
    edm.time_steps = torch.flip(torch.arange(0, DIFFUSION_STEPS, device=device), dims=[0])
    edm.T = DIFFUSION_STEPS
    edm.to(device).eval()
    for p in edm.parameters():
        p.requires_grad_(False)

x1_all, na_all, ctx_all, teacher_steps = load_pair_shards()
source_name = "teacher_pairs.x1" if SOURCE == "pairs" else EDM_WEIGHTS.name
nfe = 0 if SOURCE == "pairs" else DIFFUSION_STEPS
log(f"SOURCE={SOURCE}  nfe={nfe}  pool={na_all.shape[0]}  workers={N_WORKERS}  arom={AROMATIC_MODE}")


@torch.inference_mode()
def next_xyz_batch(start, batch):
    """Return (xyz_blocks, next_start). Wraps the pool if TARGET > n pairs."""
    n = na_all.shape[0]
    idx = (torch.arange(batch) + start) % n
    if SOURCE == "pairs":
        return x1_to_xyz_blocks(x1_all[idx], na_all[idx]), start + batch
    na = na_all[idx].to(device=device, dtype=torch.long)
    ctx = ctx_all[idx].to(device=device, dtype=torch.float32)
    nm, em = prepare_masks(na, PAD_TO, device)
    bctx = ((ctx - norms["mean"]) / norms["mad"]).unsqueeze(1).expand(-1, PAD_TO, -1) * nm
    x, h = edm(nm, em, bctx)
    mols = samples_to_rdkit_mol(x.cpu(), h.cpu(), nm.cpu(), ATOM_DECODER)
    return [Chem.MolToXYZBlock(m) for m in mols], start + batch


# --------------------- run ---------------------
shard_i = 0
while (OUT_DIR / f"shard{shard_i:04d}.pt").exists():
    shard_i += 1
kept = shard_i * SHARD_SIZE
log(f"resume at shard {shard_i} (kept≈{kept})")

buf = {k: [] for k in ("elements", "coords", "conn", "target", "n_atoms")}
seen = 0
cursor = 0
batch = BATCH_PAIRS if SOURCE == "pairs" else BATCH_EDM
pbar = tqdm(total=TARGET, initial=kept, desc="labelled")
pool = ProcessPoolExecutor(
    max_workers=N_WORKERS,
    initializer=_init_worker,
    initargs=(AROMATIC_MODE, DROP_UNSANITIZABLE, MIN_ATOMS),
)
try:
    while kept < TARGET:
        blocks, cursor = next_xyz_batch(cursor, batch)
        seen += len(blocks)
        for res in pool.map(label_xyz, blocks, chunksize=16):
            if res is None:
                continue
            e, c, cn, tg, n = res
            buf["elements"].append(e); buf["coords"].append(c)
            buf["conn"].append(cn); buf["target"].append(tg); buf["n_atoms"].append(n)
            kept += 1
            pbar.update(1)
            if len(buf["n_atoms"]) >= SHARD_SIZE:
                flush_shard(buf, shard_i, source_name, nfe)
                buf = {k: [] for k in buf}
                shard_i += 1
            if kept >= TARGET:
                break
        pbar.set_postfix(keep=f"{kept / max(seen, 1):.1%}")
finally:
    pool.shutdown()
    if buf["n_atoms"]:
        flush_shard(buf, shard_i, source_name, nfe)
    pbar.close()

log(f"done kept={kept} seen={seen} keep_rate={kept / max(seen, 1):.3f} arom={AROMATIC_MODE} source={source_name}")

In [ ]:
"""Label sanity: class balance, bonds/molecule, size coverage, visual spot-check."""
from collections import Counter
from pathlib import Path

import py3Dmol
import torch
from rdkit import Chem

try:
    from src.mlconfgen.utils.common import bond_type_dict
except ImportError:
    from mlconfgen.utils.common import bond_type_dict

OUT_DIR = Path("./bond_pairs")
shards = sorted(OUT_DIR.glob("shard*.pt"))
print(f"{len(shards)} shards")
pack = torch.load(shards[0], map_location="cpu", weights_only=False)

tgt, conn, na = pack["target"], pack["conn"], pack["n_atoms"]
n_mol = na.shape[0]
iu = torch.triu_indices(tgt.shape[1], tgt.shape[2], offset=1)
pairs = tgt[:, iu[0], iu[1]]

cnt = Counter(pairs.flatten().tolist())
total = sum(cnt.values())
names = {0: "none", 1: "single", 2: "double", 3: "triple", 4: "aromatic"}
print(f"\nmode={pack['aromatic_mode']}  source={pack.get('source')}  nfe={pack.get('nfe')}  n={n_mol}")
print("class balance over upper-triangle pairs:")
for k in range(5):
    print(f"  {k} {names[k]:9s} {cnt.get(k, 0):10d}  {cnt.get(k, 0) / total:7.4%}")

bonds = (pairs > 0).sum(1).float()
print(f"\nbonds/mol      mean={bonds.mean():.1f} min={int(bonds.min())} max={int(bonds.max())}")
print(f"atoms/mol      mean={na.float().mean():.1f} range={int(na.min())}-{int(na.max())}")

ci = conn[:, iu[0], iu[1]] > 0
ti = pairs > 0
print(f"\ninput-vs-target connectivity (the part the net must fix):")
print(f"  obabel bond missing from RDKit guess: {(ti & ~ci).sum().item() / max(ti.sum().item(), 1):.3%}")
print(f"  RDKit bond absent in obabel target:   {(ci & ~ti).sum().item() / max(ci.sum().item(), 1):.3%}")

sizes = Counter(na.tolist())
print("\nsize coverage: " + " ".join(f"{s}:{sizes.get(s, 0)}" for s in range(15, 40)))


def mol_from_labels(elements, coords, target, n):
    m = Chem.RWMol()
    for z in elements[:n].tolist():
        m.AddAtom(Chem.Atom(int(z)))
    for i in range(n):
        for j in range(i):
            o = int(target[i, j])
            if o:
                m.AddBond(j, i, bond_type_dict[o])
    mol = m.GetMol()
    conf = Chem.Conformer(n)
    for i in range(n):
        conf.SetAtomPosition(i, [float(v) for v in coords[i]])
    mol.AddConformer(conf)
    return mol


ok, smis, show_mols = 0, [], []
for i in range(min(500, n_mol)):
    mol = mol_from_labels(pack["elements"][i], pack["coords"][i].float(), tgt[i], int(na[i]))
    try:
        probe = Chem.Mol(mol)
        Chem.SanitizeMol(probe)
        ok += 1
        if len(show_mols) < 4:
            show_mols.append(mol)
        if len(smis) < 5:
            smis.append(Chem.MolToSmiles(probe))
    except Exception:
        pass
print(f"\nlabels sanitize: {ok}/{min(500, n_mol)}  (should be ~100% with DROP_UNSANITIZABLE)")
for s in smis:
    print("  ", s)

cols = max(1, len(show_mols))
v = py3Dmol.view(viewergrid=(1, cols), width=250 * cols, height=250)
for i, m in enumerate(show_mols):
    v.addModel(Chem.MolToMolBlock(m, kekulize=False), "mol", viewer=(0, i))
    v.setStyle({"stick": {"radius": 0.12}, "sphere": {"scale": 0.22}}, viewer=(0, i))
    v.zoomTo(viewer=(0, i))
v.show()